### Input

**Come usare**

per commentare, fai CTRL F 'second step'  
mi dà tre risultati: i primi due commenta sopra e sotto, l'ultimo cambia pars in init

ricorda che la matrice di covarianza mi dà il quadrato dell'incertezza, devo far la radice quadrata  

**data.txt file**

1- headers  
don't put headers in the .txt file  
but if you did, in the load_text command put: `skiprows=1`   
or `comments='#'` to tell it to skip lines starting with the char #   

2- delimiters  
se il delimiter non è uno spazio, usa: `delimiter=','` 

3- arrays are rows or cols?  
unpack=True: each var is a col (like usual)  
unpack=False: each var is a row   

4- starting dir:  
attenta, lo script non è lanciato da dove si trova, ma magari da un'altro posto  
per sicurezza, metti global path  

### Changing the graph

**Always, make graph pretty**

1- se stampo, per non tagliare le label, setta la figure:  
`pylab.figure(figsize=(20,10))`

2- from the errorbar, delete `marker='.'`  
this ways it only plots the bars, which can be seen better, not the dot  

**Grafico bilog**

1- metti log sulle assi:  
`pylab.xscale('log')`   
`pylab.yscale('log')`  

2- quando scegli i punti, prendili log:  
`xx = np.geomspace(a,b, 1000)`  
a, b sono max(x) e min(x), come in linspace  
(NON è vero che sono gli ESPONENTI delle potenze di 10 da cui iniziare e finire) 

3- per avere una retta:  
power law: bilog  
esponenziale: semilog (log scale only on y axis)  

**Seeing stuff**

where i plot someone, add the zorder params  
give it a number, 1 is the highest, it's plotted on top, it's definitely seen  
2 is lower, less seen  
zorder=importance  

ax.errorbar(x, y, Dy, Dx, linestyle='', color='black', marker='.', zorder=1)

**Add lines**

general method: using plot:  
ax.plot([3, 3], [ymin, ymax], '--', color='black')   # line at x=3, between ymin and ymax  
ax.plot([xmin, xmax], [5, 5], '--', color='black')

other methods (hline = horizontal, vline=vertical):  
ax.axhline(y=5, linestyle='--', color='blue', linewidth=1.5, alpha=0.7, label='threshold')
ax.axvline(x=3)  
ax.axvline(x=3, ymin=0.2, ymax=0.8)  # at x=3, limited between 0.2 and 0.8 in height  
ax.axhline(y=5, xmin=0.1, xmax=0.9)


**Gradi vs radianti**

np.deg2rad(45)  
np.rad2deg(np.pi/2)  

### Funzione di trasferimento complessa

In [ ]:
# to fit the gain

def rc_highpass_H(omega, fc):
    wc = 2*np.pi*fc
    s = 1j * (omega / wc)  
    T = s / (1.0 + s)
    return np.abs(T)

# to fit the phase: 

# return numpy.angle(T)

### Plotta tanti file

**Wha to do**

at the start, do a loop on all files  
give them names easy to loop hrough, or save the different parts in a 'labels' array  

**Note**

to change axis label, must use:  
ax.set_xlabel()  
ax.set_yscale("log")

In [ ]:
import pylab
import numpy
from scipy.optimize import curve_fit


Directory='/home/studentelab2/AliceGea/Esperienza_12/'  
labels = ['vuoto', 'alluminio pieno', 'alluminio profilato', 'alluminio profilato segato', 'ferro pieno', 'ferro laminato']
N = 7 # number of files 

fig, axes = pylab.subplots(nrows=N, ncols=1, constrained_layout=True, figsize = (12,2.2*N), sharex=True)

for ax, i in zip(axes, range(N)):

    Filename = f"{Directory}file{i+1}{labels[i]}"  
    x, Dx, y, Dy = pylab.loadtxt(Filename, unpack=True)
    ax.errorbar(x, y, Dy, Dx, linestyle='', color='black', marker='.', zorder=1)

    # Now, normal code for the fit  
    # then, to print results:  

    print(f'{labels[i]} : \n omega = {pars[0]} \n tau = {pars[1]} \n Q + {pars[3]*pars[2]/2} \n')

    # At the end, whrn plotting, put ax, not pylab  
    ax.plot(xx,ff(xx,*pars), color='red', zorder=2, label = labels[i])
    ax.legend(loc='upper right')

# show the plot goes at the end, NOT in the loop  

pylab.show()

In [ ]:
### To plot in different cols  


fig, axes = pylab.subplots(nrows=3, ncols=2, sharex=True, sharey=True, figsize=(12, 6), constrained_layout=True)   # put sharey as well
axes = axes.flatten()

# so now can call ax[0], ax[1] like before, don't need to call ax[0,0] useing a matrix  
# so the code is the same as before  
# the order is: [0,0], [0,1], [1,0], [1,1]... so like reading a book, first travels all first row, then goes to second row

for ax, i in zip(axes, range(N)):
    Filename = f"{Directory}file{i}.txt"
    x, Dx, y, Dy = pylab.loadtxt(Filename, unpack=True)
    ax.errorbar(x, y, Dy, Dx, linestyle='', color='black', marker='.', zorder=1)


### Altro

#### Histogram

In [ ]:
# Histogram - quanti bin voglio, auomaticamente rimuove outliers  
vm=int(np.average(y))
bons=10
plt.figure(figsize=(8,5))
counts, bins, _ = plt.hist(y,bins=bons,range=(vm-bons/2,vm+bons/2))
xxx=np.linspace(vm-bons/2+.5,vm+bons/2-.5,bons)
# ora x e y sono xxx e counts

In [ ]:
# Find outliers
# outlier sono scelti dopo il fit, col senno di poi, dal grafico dei residui
# scelgo una thresolf e se la superano li scarto
preds = f(popt) 
residuals = y - preds
threshold = 3 * np.std(residuals)
outliers = np.abs(residuals) > threshold

plt.plot(x[outliers], y[outliers], 'ro', markersize=10, label='Outliers')  # Outliers (red circles)


# Metodo 1 - da fare prima del fit, vedo se il punto è probabile secondo una distribuizione normale
media = np.mean(V)
sigma = np.std(V)
zscore = (V - media) / sigma
# criterio: |z| > 3 → outlier
mask_outliers = np.abs(zscore) > 3
# dati puliti per il fit
t_clean = t[~mask_outliers]
V_clean = V[~mask_outliers]

In [ ]:
# far partire i tempi da 0 

t = np.array([0.137, 0.142, 0.147, 0.152])
t = t - t[0]

#### Other

**Comment**  

Pyzo:  
select lines  
Ctrl 4: comment all lines  
Ctrl 5: uncomment all lines  
oppure usa le 3 apici  
lancia programma: Ctrl E  

VScode:  
select lines  
Ctrl Shift 7 : comment /uncomment selected lines

**Valori complessi**

numpy può avere numeri complessi  
basta fare moltiplicare per $1j$  
poi per calcolare il gain, basta fare np.abs(H)  
si vede il programma python nella videolezione del 5 dicembre  